##### Copyright 2025 Google LLC.

# 🚀 Multi-Agent Systems & Workflow Patterns

**Welcome to the Kaggle 5-day Agents course!**

In the previous notebook, you built a **single agent** that could take action. Now, you'll learn how to scale up by building **agent teams**.

Just like a team of people, you can create specialized agents that collaborate to solve complex problems. This is called a **multi-agent system**, and it's one of the most powerful concepts in AI agent development.

In this notebook, you'll:

- ✅ Learn when to use multi-agent systems in [Agent Development Kit (ADK)](https://google.github.io/adk-docs/)
- ✅ Build your first system using an LLM as a "manager"
- ✅ Learn three core workflow patterns (Sequential, Parallel, and Loop) to coordinate your agent teams 

## ⚙️ Section 1: Setup

### 1.1: Install dependencies

The Kaggle Notebooks environment includes a pre-installed version of the [google-adk](https://google.github.io/adk-docs/) library for Python and its required dependencies, so you don't need to install additional packages in this notebook.

To install and use ADK in your own Python development environment outside of this course, you can do so by running:

```
pip install google-adk
```

### 1.2: Configure your Gemini API Key

This notebook uses the [Gemini API](https://ai.google.dev/gemini-api/docs), which requires authentication.

**1. Get your API key**

If you don't have one already, create an [API key in Google AI Studio](https://aistudio.google.com/app/api-keys).

**2. Add the key to Kaggle Secrets**

Next, you will need to add your API key to your Kaggle Notebook as a Kaggle User Secret.

1. In the top menu bar of the notebook editor, select `Add-ons` then `Secrets`.
2. Create a new secret with the label `GOOGLE_API_KEY`.
3. Paste your API key into the "Value" field and click "Save".
4. Ensure that the checkbox next to `GOOGLE_API_KEY` is selected so that the secret is attached to the notebook.

**3. Authenticate in the notebook**

Run the cell below to complete authentication.

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


### 1.3: Import ADK components

Now, import the specific components you'll need from the Agent Development Kit and the Generative AI library. This keeps your code organized and ensures we have access to the necessary building blocks.

In [2]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4: Configure Retry Options

When working with LLMs, you may encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [3]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)
print("Configuration process has been completed.")

Configuration process has been completed.


---
## 🤔 Section 2: Why Multi-Agent Systems? + Your First Multi-Agent

**The Problem: The "Do-It-All" Agent**

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

**The Solution: A Team of Specialists**

Instead of one "do-it-all" agent, we can build a **multi-agent system**. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent *only* does research, another *only* writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

To learn more, check out the documentation related to [LLM agents in ADK](https://google.github.io/adk-docs/agents/llm-agents/).

**Architecture: Single Agent vs Multi-Agent Team**

<!--
```mermaid
graph TD
    subgraph Single["❌ Monolithic Agent"]
        A["One Agent Does Everything"]
    end

    subgraph Multi["✅ Multi-Agent Team"]
        B["Root Coordinator"] -- > C["Research Specialist"]
        B -- > E["Summary Specialist"]

        C -- >|findings| F["Shared State"]
        E -- >|summary| F
    end

    style A fill:#ffcccc
    style B fill:#ccffcc
    style F fill:#ffffcc
```
-->

<img width="800" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/multi-agent-team.png" alt="Multi-agent Team" />

### 2.1 Example: Research & Summarization System

Let's build a system with two specialized agents:

1. **Research Agent** - Searches for information using Google Search
2. **Summarizer Agent** - Creates concise summaries from research findings

In [4]:
# Research Agent: Its job is to use the google_search tool and present findings.
research_agent = Agent(
    name="ResearchAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""",
    tools=[google_search],
    output_key="research_findings",  # The result of this agent will be stored in the session state with this key.
)

print("✅ research_agent created.")

✅ research_agent created.


In [5]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

print("✅ summarizer_agent created.")

✅ summarizer_agent created.


Refer to the ADK documentation for more information on [guiding agents with clear and specific instructions](https://google.github.io/adk-docs/agents/llm-agents/).

Then we bring the agents together under a root agent, or coordinator:

In [6]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


Here we're using `AgentTool` to wrap the sub-agents to make them callable tools for the root agent. We'll explore `AgentTool` in-detail on Day 2.

Let's run the agent and ask it about a topic:

In [7]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "What are the latest advancements in Blockchain technology?"
)


 ### Created new session: debug_session_id

User > What are the latest advancements in Blockchain technology?


/usr/local/lib/python3.12/dist-packages/google/adk/flows/llm_flows/base_llm_flow.py:449: UserWarning: [EXPERIMENTAL] feature FeatureName.PROGRESSIVE_SSE_STREAMING is enabled.
  async for event in agen:


ResearchCoordinator > The latest advancements in blockchain technology are characterized by its increasing integration with other emerging technologies like AI, the tokenization of real-world assets, and enhanced scalability and interoperability solutions. There's also a growing focus on sustainability, privacy, and security, alongside the expansion of Decentralized Finance (DeFi). Furthermore, the development of Central Bank Digital Currencies (CBDCs) and the accessibility provided by Blockchain-as-a-Service (BaaS) platforms are shaping the future of blockchain adoption across various industries.


**Rendered Output:** <br/>
User > What are the latest advancements in Blockchain technology?

<br/>

/usr/local/lib/python3.12/dist-packages/google/adk/flows/llm_flows/base_llm_flow.py:449: UserWarning: [EXPERIMENTAL] feature FeatureName.PROGRESSIVE_SSE_STREAMING is enabled.
  async for event in agen:
ResearchCoordinator > The latest advancements in blockchain technology are centered around enhancing its scalability, privacy, and integration with other emerging technologies. Key developments include:

*   **Real-World Asset (RWA) Tokenization:** Digitizing traditional assets like real estate and bonds to increase their liquidity and tradability.
*   **Integration with AI:** Combining blockchain with AI to create more intelligent, transparent, and secure systems, with AI aiding in data analysis and fraud detection on the blockchain.
*   **Scalability Solutions (Layer 2):** Developing Layer 2 networks that operate on top of existing blockchains to significantly boost transaction speed and efficiency.
*   **Zero-Knowledge Proofs (ZKPs):** Advancing privacy technologies that allow for verification of information without disclosing the data itself, crucial for sensitive transactions and systems.
*   **Interoperability:** Enabling seamless communication and asset exchange between different blockchain networks.
*   **Digital Identity:** Utilizing blockchain for decentralized identification systems, giving users more control over their personal data.
*   **Sustainability:** Adopting energy-efficient consensus mechanisms like Proof of Stake (PoS) to reduce environmental impact.

These advancements are driving broader enterprise adoption and positioning blockchain as a foundational layer for digital trust and automated economies.

You've just built your first multi-agent system! You used a single "coordinator" agent to manage the workflow, which is a powerful and flexible pattern.

‼️ However, **relying on an LLM's instructions to control the order can sometimes be unpredictable.** Next, we'll explore a different pattern that gives you guaranteed, step-by-step execution.

---

## 🚥 Section 3: Sequential Workflows - The Assembly Line

**The Problem: Unpredictable Order**

The previous multi-agent system worked, but it relied on a **detailed instruction prompt** to force the LLM to run steps in order. This can be unreliable. A complex LLM might decide to skip a step, run them in the wrong order, or get "stuck," making the process unpredictable.

**The Solution: A Fixed Pipeline**

When you need tasks to happen in a **guaranteed, specific order**, you can use a `SequentialAgent`. This agent acts like an assembly line, running each sub-agent in the exact order you list them. The output of one agent automatically becomes the input for the next, creating a predictable and reliable workflow.

**Use Sequential when:** Order matters, you need a linear pipeline, or each step builds on the previous one.

To learn more, check out the documentation related to [sequential agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/).

**Architecture: Blog Post Creation Pipeline**

<!--
```mermaid
graph LR
    A["User Input: Blog about AI"] -- > B["Outline Agent"]
    B -- >|blog_outline| C["Writer Agent"]
    C -- >|blog_draft| D["Editor Agent"]
    D -- >|final_blog| E["Output"]

    style B fill:#ffcccc
    style C fill:#ccffcc
    style D fill:#ccccff
```
-->

<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/sequential-agent.png" alt="Sequential Agent" />

### 3.1 Example: Blog Post Creation with Sequential Agents

Let's build a system with three specialized agents:

1. **Outline Agent** - Creates a blog outline for a given topic
2. **Writer Agent** - Writes a blog post
3. **Editor Agent** - Edits a blog post draft for clarity and structure

In [8]:
# Outline Agent: Creates the initial blog post outline.
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Create a blog outline for the given topic with:
    1. A catchy headline
    2. An introduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought""",
    output_key="blog_outline",  # The result of this agent will be stored in the session state with this key.
)

print("✅ outline_agent created.")

✅ outline_agent created.


In [9]:
# Writer Agent: Writes the full blog post based on the outline from the previous agent.
writer_agent = Agent(
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The `{blog_outline}` placeholder automatically injects the state value from the previous agent's output.
    instruction="""Following this outline strictly: {blog_outline}
    Write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key="blog_draft",  # The result of this agent will be stored with this key.
)

print("✅ writer_agent created.")

✅ writer_agent created.


In [10]:
# Editor Agent: Edits and polishes the draft from the writer agent.
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This agent receives the `{blog_draft}` from the writer agent's output.
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the text by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog",  # This is the final output of the entire pipeline.
)

print("✅ editor_agent created.")

✅ editor_agent created.


Then we bring the agents together under a sequential agent, which runs the agents in the order that they are listed:

In [11]:
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print("✅ Sequential Agent created.")

✅ Sequential Agent created.


Let's run the agent and give it a topic to write a blog post about:

In [12]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post about the benefits of using Blockchain technology"
)


 ### Created new session: debug_session_id

User > Write a blog post about the benefits of using Blockchain technology
OutlineAgent > Here's a blog outline about the benefits of using Blockchain technology:

## Headline: Beyond Bitcoin: Unlocking a World of Benefits with Blockchain Technology

**Introduction Hook:** You've probably heard of Bitcoin, but the revolutionary technology behind it, blockchain, is so much more than just a digital currency. Imagine a system that can make transactions more secure, transparent, and efficient – that's the promise of blockchain. Let's dive into how this groundbreaking innovation is reshaping industries and what benefits it can offer you.

---

### **1. Enhanced Security and Trust: The Immutability Advantage**

*   **Tamper-Proof Records:** Once data is added to a blockchain, it's incredibly difficult to alter or delete. This immutability ensures the integrity of records, building trust in the system.
*   **Decentralized Control:** Instead of a si

**Rendered Output:** <br/>

### Created new session: debug_session_id

User > Write a blog post about the benefits of using Blockchain technology
OutlineAgent > Here is a blog outline about the benefits of using Blockchain technology:

### Blog Post Outline:

**Headline:** Beyond Bitcoin: Unlocking the Revolutionary Benefits of Blockchain Technology

**Introduction Hook:** Imagine a world where transactions are instant, secure, and transparent, with no single point of failure or control. This isn't a far-off sci-fi dream; it's the reality that blockchain technology is rapidly ushering in. While often associated with cryptocurrencies like Bitcoin, blockchain's potential extends far beyond digital money, offering a powerful suite of benefits that are reshaping industries.

**Main Sections:**

**1. Enhanced Security and Immutability: The Unbreakable Ledger**
    *   **Decentralized and Distributed:** Data is stored across a network of computers, making it incredibly difficult for hackers to compromise.
    *   **Cryptographic Security:** Each block of data is cryptographically linked to the previous one, creating a tamper-proof chain. Any alteration is immediately detectable.
    *   **Immutability of Records:** Once data is added to the blockchain, it cannot be altered or deleted, ensuring a permanent and verifiable audit trail.

**2. Unprecedented Transparency and Trust: Eliminating the Middleman**
    *   **Shared, Public Ledger:** Transactions are visible to all participants on the network, fostering accountability and reducing the need for intermediaries.
    *   **Increased Trust:** Because data is verifiable and tamper-proof, trust is built directly into the system, reducing reliance on third-party verification.
    *   **Streamlined Auditing:** The transparent nature of blockchain simplifies auditing processes, saving time and resources.

**3. Increased Efficiency and Reduced Costs: Faster, Cheaper Transactions**
    *   **Disintermediation:** Eliminating intermediaries like banks or clearinghouses in transactions leads to lower fees and faster processing times.
    *   **Automation with Smart Contracts:** Self-executing contracts that automatically fulfill terms when conditions are met can automate complex processes, reducing manual effort and errors.
    *   **Global Accessibility:** Blockchain facilitates borderless transactions, making it easier and cheaper to send value across different regions.

**4. Empowering New Business Models and Innovations**
    *   **Supply Chain Management:** Tracking goods from origin to destination with verifiable records ensures authenticity and transparency.
    *   **Digital Identity and Ownership:** Securely managing personal data and ownership of digital assets.
    *   **Decentralized Finance (DeFi):** Creating financial services without traditional institutions, offering greater access and control.

**Concluding Thought:** Blockchain technology is more than just a trend; it's a foundational shift in how we can record, share, and trust information. By embracing its inherent security, transparency, and efficiency, businesses and individuals alike can unlock new opportunities, build stronger systems, and ultimately, participate in a more trustworthy and interconnected digital future.

WriterAgent > ## Beyond Bitcoin: Unlocking the Revolutionary Benefits of Blockchain Technology

Imagine a world where transactions are instant, secure, and transparent, with no single point of failure or control. This isn't a far-off sci-fi dream; it's the reality that blockchain technology is rapidly ushering in. While often associated with cryptocurrencies like Bitcoin, blockchain's potential extends far beyond digital money, offering a powerful suite of benefits that are reshaping industries.

At its core, blockchain is an **unbreakable ledger**. Its decentralized and distributed nature, coupled with cryptographic security, makes data incredibly resistant to tampering. Once information is recorded, it becomes immutable – a permanent, verifiable audit trail that fosters **unprecedented transparency and trust**. This eliminates the need for traditional intermediaries, streamlining processes and significantly **increasing efficiency and reducing costs**. Think faster, cheaper, borderless transactions, all thanks to the elimination of middlemen and the power of smart contracts.

But the benefits don't stop there. Blockchain is empowering entirely new business models. From ensuring the authenticity of goods in complex supply chains to securely managing digital identities and revolutionizing finance through Decentralized Finance (DeFi), its applications are vast and growing.

Blockchain technology is a foundational shift in how we record, share, and trust information. By embracing its inherent security, transparency, and efficiency, we can unlock new opportunities and build stronger, more trustworthy digital systems for everyone.

EditorAgent > ## Beyond Bitcoin: Unlocking the Revolutionary Benefits of Blockchain Technology

Imagine a world where transactions are instant, secure, and transparent, with no single point of failure or control. This isn't a far-off sci-fi dream; it's the reality that blockchain technology is rapidly ushering in. While often associated with cryptocurrencies like Bitcoin, blockchain's potential extends far beyond digital money, offering a powerful suite of benefits that are reshaping industries.

At its core, blockchain is an **unbreakable ledger**. Its decentralized and distributed nature, coupled with cryptographic security, makes data incredibly resistant to tampering. Once information is recorded, it becomes immutable – a permanent, verifiable audit trail that fosters **unprecedented transparency and trust**. This eliminates the need for traditional intermediaries, streamlining processes and significantly **increasing efficiency and reducing costs**. Think faster, cheaper, borderless transactions, all thanks to the elimination of middlemen and the power of smart contracts.

But the benefits don't stop there. Blockchain is empowering entirely new business models. From ensuring the authenticity of goods in complex supply chains to securely managing digital identities and revolutionizing finance through Decentralized Finance (DeFi), its applications are vast and growing.

Blockchain technology is a foundational shift in how we record, share, and trust information. By embracing its inherent security, transparency, and efficiency, we can unlock new opportunities and build stronger, more trustworthy digital systems for everyone.

👏 Great job! You've now created a reliable "assembly line" using a sequential agent, where each step runs in a predictable order.

**This is perfect for tasks that build on each other, but it's slow if the tasks are independent.** Next, we'll look at how to run multiple agents at the same time to speed up your workflow.

---
## 🛣️ Section 4: Parallel Workflows - Independent Researchers

**The Problem: The Bottleneck**

The previous sequential agent is great, but it's an assembly line. Each step must wait for the previous one to finish. What if you have several tasks that are **not dependent** on each other? For example, researching three *different* topics. Running them in sequence would be slow and inefficient, creating a bottleneck where each task waits unnecessarily.

**The Solution: Concurrent Execution**

When you have independent tasks, you can run them all at the same time using a `ParallelAgent`. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

**Use Parallel when:** Tasks are independent, speed matters, and you can execute concurrently.

To learn more, check out the documentation related to [parallel agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/parallel-agents/).

**Architecture: Multi-Topic Research**

<!--
```mermaid
graph TD
    A["User Request: Research 3 topics"] -- > B["Parallel Execution"]
    B -- > C["Tech Researcher"]
    B -- > D["Health Researcher"]
    B -- > E["Finance Researcher"]

    C -- > F["Aggregator"]
    D -- > F
    E -- > F
    F -- > G["Combined Report"]

    style B fill:#ffffcc
    style F fill:#ffccff
```
-->

<img width="600" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/parallel-agent.png" alt="Parallel Agent" />

### 4.1 Example: Parallel Multi-Topic Research

Let's build a system with four agents:

1. **Tech Researcher** - Researches AI/ML news and trends
2. **Health Researcher** - Researches recent medical news and trends
3. **Finance Researcher** - Researches finance and fintech news and trends
4. **Aggregator Agent** - Combines all research findings into a single summary

In [13]:
# Tech Researcher: Focuses on AI and ML trends.
tech_researcher = Agent(
    name="TechResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research the latest AI/ML trends. Include 3 key developments,
the main companies involved, and the potential impact. Keep the report very concise (100 words).""",
    tools=[google_search],
    output_key="tech_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech_researcher created.")

✅ tech_researcher created.


In [14]:
# Health Researcher: Focuses on medical breakthroughs.
health_researcher = Agent(
    name="HealthResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research recent medical breakthroughs. Include 3 significant advances,
their practical applications, and estimated timelines. Keep the report concise (100 words).""",
    tools=[google_search],
    output_key="health_research",  # The result will be stored with this key.
)

print("✅ health_researcher created.")

✅ health_researcher created.


In [15]:
# Finance Researcher: Focuses on fintech trends.
finance_researcher = Agent(
    name="FinanceResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research current fintech trends. Include 3 key trends,
their market implications, and the future outlook. Keep the report concise (100 words).""",
    tools=[google_search],
    output_key="finance_research",  # The result will be stored with this key.
)

print("✅ finance_researcher created.")

✅ finance_researcher created.


In [16]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.
aggregator_agent = Agent(
    name="AggregatorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction="""Combine these three research findings into a single executive summary:

    **Technology Trends:**
    {tech_research}
    
    **Health Breakthroughs:**
    {health_research}
    
    **Finance Innovations:**
    {finance_research}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. The final summary should be around 200 words.""",
    output_key="executive_summary",  # This will be the final output of the entire system.
)

print("✅ aggregator_agent created.")

✅ aggregator_agent created.


👉 **Then we bring the agents together under a parallel agent, which is itself nested inside of a sequential agent.**

This design ensures that the research agents run first in parallel, then once all of their research is complete, the aggregator agent brings together all of the research findings into a single report:

In [17]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher, health_researcher, finance_researcher],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print("✅ Parallel and Sequential Agents created.")

✅ Parallel and Sequential Agents created.


Let's run the agent and give it a prompt to research the given topics:

In [18]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Run the daily executive briefing on Tech, Health, and Finance"
)


 ### Created new session: debug_session_id

User > Run the daily executive briefing on Tech, Health, and Finance
HealthResearcher > Here's a summary of recent breakthroughs in Tech, Health, and Finance:

**Tech:** AI is increasingly integrated across industries, enhancing creativity and automating tasks. Wi-Fi 7 offers significantly faster connectivity. Green energy advancements include improved battery technology and the adoption of green hydrogen. Quantum computing is seeing tangible progress, promising to solve complex problems.

**Health:** New antipsychotic drugs targeting muscarinic receptors, like KarXT, are emerging. Precision cancer treatments, such as Osimertinib, are showing extended patient survival. GLP-1 drugs like Tirzepatide are effective for weight loss and diabetes management. Advancements in brain-computer interfaces enable communication for paralyzed individuals.

**Finance:** AI is revolutionizing finance through data analysis for risk assessment and personalized 

**Rendered Output:** <br/>

### Created new session: debug_session_id

User > Run the daily executive briefing on Tech, Health, and Finance
TechResearcher > **Key AI/ML Trends & Impact**

**1. Generative AI's Creative Leap:** Generative AI, capable of creating text, images, and code, is revolutionizing content creation and accelerating innovation. Major players like Google (with models like PaLM and DeepMind's Gato) and OpenAI are at the forefront. This trend empowers businesses to automate content generation, assist developers, and deliver highly personalized customer experiences, potentially lowering operational costs.

**2. Intelligent Automation Expansion:** Intelligent Process Automation (IPA) is merging Robotic Process Automation (RPA) with AI for end-to-end business process automation. This trend boosts productivity, efficiency, and accuracy across industries. Companies are leveraging IPA to automate repetitive tasks, especially those involving unstructured data, driving digital transformation.

**3. Ethical and Explainable AI (XAI) Growth:** As AI becomes more pervasive, the demand for ethical and transparent AI systems is rising. XAI aims to demystify complex "black box" models, fostering trust and ensuring fair treatment. This is crucial for responsible AI adoption, particularly in sensitive sectors like finance and healthcare, where bias mitigation and transparent decision-making are paramount.

These advancements are poised to reshape industries, redefine job roles, and fundamentally alter decision-making processes across the global economy.
HealthResearcher > **Health Breakthroughs:**

1.  **CRISPR Gene Editing:** Practical applications include correcting genetic defects for diseases like sickle cell anemia and thalassemia. Further development could lead to cures for inherited disorders. Timeline: Ongoing, with clinical applications expanding rapidly over the next 3-5 years.
2.  **mRNA Vaccine Technology:** Beyond COVID-19, this technology is being explored for other infectious diseases and cancers. It offers rapid development and adaptable platforms. Timeline: Already in use, with expanded applications anticipated within 2-5 years.
3.  **Protein Biomarker Analysis:** Enables earlier and more accurate cancer detection through blood tests. This could revolutionize cancer screening and treatment. Timeline: Early stages, with broader clinical adoption expected in 5-10 years.

**Technology Breakthroughs:**

1.  **Edge AI and 5G Expansion:** Enables devices to process AI locally, improving speed, privacy, and efficiency. This is crucial for smart cities and personalized experiences. Timeline: Rapidly expanding now, with widespread integration within 1-3 years.
2.  **Quantum Computing:** Significant progress is being made in developing stable quantum systems. Practical applications could include complex problem-solving in drug discovery and materials science. Timeline: Early commercial applications emerging in 5-10 years.
3.  **Generative AI and AI Agents:** Beyond chatbots, AI agents can now plan and execute tasks. This is transforming workflows in coding, writing, and customer service. Timeline: Already impacting industries, with advanced capabilities expected in 2-5 years.

**Finance Breakthroughs:**

1.  **AI-Powered Personalized Banking:** AI analyzes spending patterns and provides tailored financial guidance, proactive fraud detection, and automated savings. Timeline: Currently being implemented, with advanced features becoming standard in 1-3 years.
2.  **Tokenized Assets & Fractional Ownership:** Allows for fractional ownership of real-world assets like real estate and art, creating new investment opportunities. Timeline: Growing rapidly, with significant market expansion projected by 2030.
3.  **IoT Payment Networks:** Connected devices facilitate contactless payments and automated claims, creating an interconnected financial ecosystem. Timeline: Poised for explosive growth, with billions of connected devices expected by 2025.
FinanceResearcher > **Executive Briefing: Tech, Health, and Finance Trends for 2026**

**Key Fintech Trends:**

1.  **AI-Native Fintechs & Agentic AI:** AI is deeply embedded in new fintechs, enabling autonomous decision-making and task execution for transactions and payments. Market implication: Increased efficiency, personalized services, and a shift towards agent-driven commerce.
2.  **Stablecoin Infrastructure & Digital Assets:** Stablecoins are gaining traction as settlement infrastructure, driven by regulatory clarity and expanding use cases. Market implication: Improved cross-border transactions, greater investor interest, and a focus on governance.
3.  **Fintech Consolidation & Focus on Profitability:** Increased M&A activity and a shift from broad offerings to deep, industry-specific solutions driven by a focus on profitability. Market implication: Market maturation, strategic acquisitions, and a clearer differentiation between successful and struggling firms.

**Key Health Tech Trends:**

1.  **AI in Diagnostics and Personalized Care:** AI is increasingly used to aid diagnosis, personalize treatment plans, and support administrative workflows. Market implication: Earlier disease detection, tailored healthcare, and improved operational efficiency.
2.  **IoMT & Connected Health Ecosystems:** The Internet of Medical Things (IoMT) is foundational for real-time data sharing, enabling better care coordination and faster decision-making. Market implication: Proactive and continuous patient monitoring, streamlined hospital operations, and integrated care platforms.
3.  **Remote Monitoring and Telehealth Integration:** Remote patient monitoring and telehealth are becoming integral to chronic care management and general healthcare delivery. Market implication: Enhanced patient access, proactive intervention for chronic conditions, and care extending beyond traditional clinical settings.

**Key Technology Trends:**

1.  **AI Everywhere & AI Supercomputing:** AI is becoming the backbone of business, with advancements in AI Supercomputing platforms enabling complex model training. Market implication: Pervasive AI integration across industries, driving innovation and requiring robust infrastructure.
2.  **Cybersecurity Reinvented & AI-Powered Cybersecurity:** Enhanced cybersecurity measures are critical due to AI-powered threats and evolving attack landscapes. Market implication: Increased investment in advanced security solutions, proactive threat detection, and a focus on data privacy.
3.  **Cloud Maturation & Geopatriation:** Cloud adoption is shifting towards strategic optimization and value creation. Geopatriation emphasizes localizing tech infrastructure to mitigate geopolitical risks. Market implication: Hybrid and multi-cloud strategies, cloud-native applications, and a focus on regional tech sovereignty.

**Future Outlook:**

The coming years will see significant technological integration across finance and health, driven by AI. Fintech will focus on agentic capabilities and digital asset infrastructure, while health tech will emphasize AI-driven personalization and connected care. Across all sectors, robust cybersecurity and strategic cloud adoption will be paramount. These trends point towards a future of more efficient, personalized, and secure services, albeit with increasing complexity and regulatory considerations.
AggregatorAgent > ## Executive Summary: Convergence of AI, Health, and Finance

Artificial Intelligence (AI) is the unifying force across technology, health, and finance, driving unprecedented innovation and operational efficiency. Generative AI and intelligent automation are rapidly transforming content creation, business processes, and customer experiences, while advancements in Edge AI and 5G are enhancing speed and privacy.

In healthcare, AI is revolutionizing diagnostics and personalized care, complementing breakthroughs in gene editing (CRISPR) and mRNA vaccine technology. The integration of the Internet of Medical Things (IoMT) and telehealth is creating connected, proactive health ecosystems.

The financial sector is experiencing a similar AI-driven transformation, with AI-powered personalized banking, autonomous agentic capabilities in fintech, and the emergence of tokenized assets. This convergence points to a future of highly personalized, efficient, and secure services across all sectors. However, the increasing pervasiveness of AI necessitates a strong emphasis on ethical AI development and robust cybersecurity measures to ensure trust and mitigate evolving threats.

🎉 Great! You've seen how parallel agents can dramatically speed up workflows by running independent tasks concurrently.

So far, all our workflows run from start to finish and then stop. **But what if you need to review and improve an output multiple times?** Next, we'll build a workflow that can loop and refine its own work.

---
## ➰ Section 5: Loop Workflows - The Refinement Cycle

**The Problem: One-Shot Quality**

All the workflows we've seen so far run from start to finish. The `SequentialAgent` and `ParallelAgent` produce their final output and then stop. This 'one-shot' approach isn't good for tasks that require refinement and quality control. What if the first draft of our story is bad? We have no way to review it and ask for a rewrite.

**The Solution: Iterative Refinement**

When a task needs to be improved through cycles of feedback and revision, you can use a `LoopAgent`. A `LoopAgent` runs a set of sub-agents repeatedly *until a specific condition is met or a maximum number of iterations is reached.* This creates a refinement cycle, allowing the agent system to improve its own work over and over.

**Use Loop when:** Iterative improvement is needed, quality refinement matters, or you need repeated cycles.

To learn more, check out the documentation related to [loop agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/loop-agents/).

**Architecture: Story Writing & Critique Loop**

<!--
```mermaid
graph TD
    A["Initial Prompt"] -- > B["Writer Agent"]
    B -- >|story| C["Critic Agent"]
    C -- >|critique| D{"Iteration < Max<br>AND<br>Not Approved?"}
    D -- >|Yes| B
    D -- >|No| E["Final Story"]

    style B fill:#ccffcc
    style C fill:#ffcccc
    style D fill:#ffffcc
```
-->

<img width="250" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/loop-agent.png" alt="Loop Agent" />

### 5.1 Example: Iterative Story Refinement

Let's build a system with two agents:

1. **Writer Agent** - Writes a draft of a short story
2. **Critic Agent** - Reviews and critiques the short story to suggest improvements

In [19]:
# This agent runs ONCE at the beginning to create the first draft.
initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Based on the user's prompt, write the first draft of a short story (around 100-150 words).
    Output only the story text, with no introduction or explanation.""",
    output_key="current_story",  # Stores the first draft in the state.
)

print("✅ initial_writer_agent created.")

✅ initial_writer_agent created.


In [20]:
# This agent's only job is to provide feedback or the approval signal. It has no tools.
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a constructive story critic. Review the story provided below.
    Story: {current_story}
    
    Evaluate the story's plot, characters, and pacing.
    - If the story is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",
    output_key="critique",  # Stores the feedback in the state.
)

print("✅ critic_agent created.")

✅ critic_agent created.


Now, we need a way for the loop to actually stop based on the critic's feedback. The `LoopAgent` itself doesn't automatically know that "APPROVED" means "stop."

We need an agent to give it an explicit signal to terminate the loop.

We do this in two parts:

1. A simple Python function that the `LoopAgent` understands as an "exit" signal.
2. An agent that can call that function when the right condition is met.

First, you'll define the `exit_loop` function:

In [21]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")

✅ exit_loop function created.


To let an agent call this Python function, we wrap it in a `FunctionTool`. Then, we create a `RefinerAgent` that has this tool.

👉 **Notice its instructions:** this agent is the "brain" of the loop. It reads the `{critique}` from the `CriticAgent` and decides whether to (1) call the `exit_loop` tool or (2) rewrite the story.

In [22]:
# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.""",
    output_key="current_story",  # It overwrites the story with the new, refined version.
    tools=[
        FunctionTool(exit_loop)
    ],  # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")

✅ refiner_agent created.


Then we bring the agents together under a loop agent, which is itself nested inside of a sequential agent.

This design ensures that the system first produces an initial story draft, then the refinement loop runs up to the specified number of `max_iterations`:

In [23]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,  # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: Initial Write -> Refinement Loop.
root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


Let's run the agent and give it a topic to write a short story about:

In [24]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a short story about a lighthouse keeper who discovers a mysterious, glowing map"
)


 ### Created new session: debug_session_id

User > Write a short story about a lighthouse keeper who discovers a mysterious, glowing map
InitialWriterAgent > Elias traced the salt-crusted lines on his worn chart. The storm had raged for three days, a ceaseless battering against the stone tower. When the dawn finally broke, it revealed a splintered crate bobbing near the shore. Inside, nestled amongst sodden rope, was a rolled parchment. It felt impossibly smooth, cool to the touch. Unfurling it, Elias gasped. It wasn't paper, but something akin to polished shell. And it glowed. Not with a lamp's light, but an inner luminescence that pulsed with a soft, cerulean hue. Strange symbols, unlike any he knew, swirled across its surface, forming what looked like a map of the very stars, yet also of the seabed below. A chill, not of the sea air, prickled his skin.
CriticAgent > The story sets up an intriguing mystery and a compelling atmosphere. Here are a few suggestions for improvement:

1. 

**Rendered Output:** <br/>

### Created new session: debug_session_id

User > Write a short story about a lighthouse keeper who discovers a mysterious, glowing map
InitialWriterAgent > Old Silas, keeper of the Black Rock light for thirty years, had seen his share of strange flotsam. But nothing prepared him for the rolled parchment that washed ashore one fog-choked morning. It was thick, ancient, and when unfurled by the lamp's sputtering glow, it pulsed with an inner luminescence. Not ink, but starlight seemed to trace intricate coastlines and cryptic symbols he’d never encountered. It depicted a sea far beyond his charts, dotted with islands that shimmered with an unearthly blue. His calloused finger traced a jagged path, a tremor of both fear and exhilaration running through him. The map, it seemed, wasn't just a guide; it was an invitation.
CriticAgent > This is a promising start to a story with a clear hook. Here are a few suggestions for improvement:

1.  **Expand on Silas's Character:** While we know he's a seasoned lighthouse keeper, we don't get much insight into his personality beyond that. What are his motivations? Is he lonely, adventurous, resigned? A brief glimpse into his internal life before finding the map would make his reaction to it more impactful. For example, a sentence about his quiet routine or a longing for something more could set the stage.

2.  **Describe the Map's "Invitation" More Explicitly:** The line "The map, it seemed, wasn't just a guide; it was an invitation" is intriguing, but the story could benefit from showing *how* it's an invitation. Does Silas feel a pull towards a specific island? Does a symbol pulse brighter when he touches it? Does he hear a whisper? Making the invitation more tangible will deepen the mystery and propel the plot forward.

3.  **Hint at the "Strange Flotsam" to Contrast:** You mention Silas has seen "strange flotsam." Briefly hinting at one or two examples of what he's encountered before would highlight just how unique this glowing map truly is. Was it a message in a bottle with nonsensical writing, or a perfectly carved wooden bird? This contrast would emphasize the map's otherworldly nature.
RefinerAgent > Old Silas, keeper of the Black Rock light for thirty years, had known only the rhythmic sweep of the beam and the lonely cry of gulls. His days were a quiet routine of polished brass and well-worn logbooks, a life resigned to solitude, yet sometimes, in the dead of night, a flicker of longing for a horizon beyond his own would stir. He’d seen his share of strange flotsam – a barnacle-encrusted figurehead of a forgotten queen, a sealed bottle containing only a single, dried rose – but nothing prepared him for the rolled parchment that washed ashore one fog-choked morning.

It was thick, ancient, and when unfurled by the lamp's sputtering glow, it pulsed with an inner luminescence. Not ink, but starlight seemed to trace intricate coastlines and cryptic symbols he’d never encountered. It depicted a sea far beyond his charts, dotted with islands that shimmered with an unearthly blue. His calloused finger traced a jagged path, and as it hovered over a pulsating island marked with a spiral, a faint, melodic hum resonated in his bones, a whisper that felt like a forgotten memory. The map, it seemed, wasn't just a guide; it was an invitation, a siren song from a place that called to the hidden adventurer within him. A tremor of both fear and exhilaration ran through him, the quiet routine of his life suddenly feeling like a cage he hadn't realized he was in.

CriticAgent > APPROVED

You've now implemented a loop agent, creating a sophisticated system that can iteratively review and improve its own output. This is a key pattern for ensuring high-quality results.

You now have a complete toolkit of workflow patterns. Let's put it all together and review how to choose the right one for your use case.

--- 
## ✨ Section 6: Summary - Choosing the Right Pattern

### Decision Tree: Which Workflow Pattern?

<!--
```mermaid
graph TD
    A{"What kind of workflow do you need?"} -- > B["Fixed Pipeline<br>(A → B → C)"];
    A -- > C["Concurrent Tasks<br>(Run A, B, C all at once)"];
    A -- > D["Iterative Refinement<br>(A ⇆ B)"];
    A -- > E["Dynamic Decisions<br>(Let the LLM decide what to do)"];

    B -- > B_S["Use <b>SequentialAgent</b>"];
    C -- > C_S["Use <b>ParallelAgent</b>"];
    D -- > D_S["Use <b>LoopAgent</b>"];
    E -- > E_S["Use <b>LLM Orchestrator</b><br>(Agent with other agents as tools)"];

    style B_S fill:#f9f,stroke:#333,stroke-width:2px
    style C_S fill:#ccf,stroke:#333,stroke-width:2px
    style D_S fill:#cff,stroke:#333,stroke-width:2px
    style E_S fill:#cfc,stroke:#333,stroke-width:2px
```
-->

<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/agent-decision-tree.png" alt="Agent Decision Tree" />

### Quick Reference Table

| Pattern | When to Use | Example | Key Feature |
|---------|-------------|---------|-------------|
| **LLM-based (sub_agents)** | Dynamic orchestration needed | Research + Summarize | LLM decides what to call |
| **Sequential** | Order matters, linear pipeline | Outline → Write → Edit | Deterministic order |
| **Parallel** | Independent tasks, speed matters | Multi-topic research | Concurrent execution |
| **Loop** | Iterative improvement needed | Writer + Critic refinement | Repeated cycles |

---

## ✅ Congratulations! You're Now an Agent Orchestrator

In this notebook, you made the leap from a single agent to a **multi-agent system**.

You saw **why** a team of specialists is easier to build and debug than one "do-it-all" agent. Most importantly, you learned how to be the **director** of that team.

You used `SequentialAgent`, `ParallelAgent`, and `LoopAgent` to create deterministic workflows, and you even used an LLM as a 'manager' to make dynamic decisions. You also mastered the "plumbing" by using `output_key` to pass state between agents and make them collaborative.

**ℹ️ Note: No submission required!**

This notebook is for your hands-on practice and learning only. You **do not** need to submit it anywhere to complete the course.

### 📚 Learn More

Refer to the following documentation to learn more:

- [Agents in ADK](https://google.github.io/adk-docs/agents/)
- [Sequential Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Parallel Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/parallel-agents/)
- [Loop Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/loop-agents/)
- [Custom Agents in ADK](https://google.github.io/adk-docs/agents/custom-agents/)

### 🎯 Next Steps

Ready for the next challenge? Stay tuned for Day 2 notebooks where we'll learn how to create **Custom Functions, use MCP Tools** and manage **Long-Running operations!**

---

| Authors |
| --- |
| [Kristopher Overholt](https://www.linkedin.com/in/koverholt) |